# Hospital readmission risk for diabetes patients (Machine Learning Zoomcamp, Capstone 2)

**Question.** At the moment a hospitalised diabetes patient is discharged, how likely is it that the patient comes back, and how soon?
The target has three classes: **NO** (not readmitted), **>30** (readmitted after more than 30 days) and **<30** (readmitted within 30 days).
The early readmission `<30` is the class hospitals care about most: it is the one that readmission-reduction programmes try to prevent.

**Data.** UCI *Diabetes 130-US Hospitals for Years 1999-2008* (101,766 encounters of 71,518 patients, 50 columns, CC BY 4.0, doi:10.24432/C5230J).

**This notebook** covers data quality and cleaning, a patient-aware validation scheme, exploratory analysis of the training patients only, four model families
(logistic regression, decision tree, random forest, histogram gradient boosting), hyper-parameter tuning, model selection by cross-validated log-loss,
a one-shot evaluation on hold-out patients, a subgroup audit, permutation importance, a demonstration of patient leakage, and finally writes `params.json`, which `train.py` reads.

**How to run.** From the project folder, with the environment of `requirements-dev.txt` active:
`python -m nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 notebook.ipynb` (about 10 to 30 minutes, silent until it is done).
All shared code lives in `common.py`, so that the notebook, `train.py` and the web service cannot drift apart.

In [ ]:
import json
import os
import time
import warnings
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))   # silences a joblib warning on Windows
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import display
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score

import common

# CAPSTONE_FAST=1 shrinks every search and subsample: it is only used to smoke-test this notebook, never for the real run.
FAST = os.environ.get("CAPSTONE_FAST") == "1"
N_JOBS = 1 if FAST else -1
RS = common.RANDOM_STATE
DATA_PATH = Path("data") / common.CSV_NAME
PARAMS_PATH = Path("params_fast.json") if FAST else Path("params.json")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__} | fast mode: {FAST}")

## 1. Load the data

The file is read as text first (see `common.load_raw`): pandas would otherwise turn the value `None` (lab test not performed) and `NO` (not readmitted) into missing values.
The file's own missing-value marker is `?`; it is converted to a real missing value.

In [ ]:
if not DATA_PATH.exists():
    import download_data
    download_data.main([])
raw = common.load_raw(DATA_PATH)
print(f"{len(raw):,} encounters, {raw.shape[1]} columns, {raw['patient_nbr'].nunique():,} patients")
display(raw.head(3).T)

## 2. Data quality checks

What is missing, what is constant, what is duplicated, and how are the classes distributed?

In [ ]:
missing = raw.isna().mean().mul(100).round(2)
missing = missing[missing > 0].sort_values(ascending=False)
display(missing.to_frame("% missing (the '?' marker)"))

print("Duplicate rows:", int(raw.duplicated().sum()), "| repeated encounter ids:", int(raw["encounter_id"].duplicated().sum()))
constant = [c for c in common.MED_COLUMNS if raw[c].nunique() <= 1]
print("Drug columns with a single value:", constant)
print("Rare drug columns (a non-'No' value in fewer than 0.1 % of the encounters):",
      [c for c in common.MED_COLUMNS if (raw[c] != "No").mean() < 0.001])

counts = raw["readmitted"].value_counts().reindex(common.CLASS_LABELS)
display(pd.DataFrame({"encounters": counts, "share": (counts / len(raw)).round(3)}))

**Findings.** `weight` is missing for about 97 % of the encounters, so it is dropped. `payer_code` (about 40 %) and `medical_specialty` (about 49 %) are missing for a large share
but still informative: *missing* becomes its own level. `race`, `diag_1`, `diag_2`, `diag_3` have only a few missing values.
Two drug columns never vary and several others almost never do. The classes are imbalanced: the early readmission `<30` is only about one encounter in nine.

In [ ]:
per_patient = raw.groupby("patient_nbr").size()
multi = per_patient[per_patient > 1]
print(f"Patients: {len(per_patient):,}. Patients with more than one encounter: {len(multi):,} ({len(multi) / len(per_patient):.1%}).")
print(f"Encounters that belong to a patient with more than one encounter: {multi.sum():,} ({multi.sum() / len(raw):.1%} of all rows).")
display(per_patient.value_counts().sort_index().head(8).rename("patients").to_frame().T.rename_axis(columns="encounters per patient"))

**Why this matters.** A large share of the encounters (see the figures printed above) belongs to a patient who has other encounters in the table. A random row split would put a patient's earlier stay in the training set and the next stay in the test set,
and the model could profit from recognising the patient. The validation scheme below therefore keeps **all encounters of a patient on the same side** of every split (section 4); section 10 measures how much a random split would flatter the model.

In [ ]:
display(raw[common.COUNT_INPUTS].describe().T.round(2))

In [ ]:
for column in ["age", "gender", "race", "admission_type_id", "discharge_disposition_id", "admission_source_id",
               "payer_code", "medical_specialty", "max_glu_serum", "A1Cresult", "change", "diabetesMed"]:
    counts = raw[column].value_counts(dropna=False)
    top = ", ".join(f"{'missing' if pd.isna(k) else k} {v / len(raw):.1%}" for k, v in counts.head(5).items())
    print(f"{column:<26s} {raw[column].nunique(dropna=False):>3d} levels | most frequent: {top}")

## 3. Cleaning rules

`common.clean_with_report` applies the rules below. Each one is a visible number, not a silent step.

1. **Exact duplicate rows and repeated encounter ids are dropped.** There are none in the original file; the rule protects against a damaged copy.
2. **Encounters that ended in death or hospice care are dropped** (`discharge_disposition_id` 11, 13, 14, 19, 20, 21 in `IDS_mapping.csv`). Those patients cannot be readmitted, so they are not examples of "not readmitted"; keeping them would teach the model that being close to death means low risk.
3. **Rows without a patient id, a target or a count are dropped** (none expected).
4. Several encounters of the same patient are **kept** (they are real events) but never split across train and test.

Deliberate modelling decisions that are not row filters (all implemented in `common.build_features`):

* `weight` (97 % missing) and the two identifiers are not used.
* **`race` and `gender` are not model inputs.** They are kept in the cleaned table only so that section 9 can audit the model's behaviour across groups.
* Diagnosis codes are grouped into nine ICD-9 families (Strack et al., 2014); 'not recorded' payer or specialty becomes the level `Unknown`.
* Not-available / NULL / not-mapped ids of admission type, admission source and discharge disposition are merged into one level `unknown`; categories with fewer than 100 training rows are pooled into one level by the encoder.
* The 23 drug columns become the number of active drugs, the number of drugs whose dose was changed, and the two most important drugs (`insulin`, `metformin`).

In [ ]:
data, report = common.clean_with_report(raw)
print(json.dumps(report, indent=2))

## 4. Validation scheme

* **Hold-out test set:** one fold of a stratified, patient-grouped 5-fold split, about 20 % of the patients. It is looked at **once**, in section 8.
* **Cross-validation, tuning and selection** use the training patients only, with `StratifiedGroupKFold` (folds keep the class proportions and never split a patient).
* **EDA** below also uses the training patients only, so that no decision is informed by the test set.

In [ ]:
X_all, y_all = data[common.INPUT_COLUMNS], data[common.TARGET]
groups_all = data[common.GROUP_COLUMN].to_numpy()
train_mask, test_mask = common.grouped_stratified_split(y_all, groups_all)

train_df = data.loc[train_mask].reset_index(drop=True)
test_df = data.loc[test_mask].reset_index(drop=True)
X_train, y_train, g_train = train_df[common.INPUT_COLUMNS], train_df[common.TARGET], train_df[common.GROUP_COLUMN].to_numpy()
X_test, y_test = test_df[common.INPUT_COLUMNS], test_df[common.TARGET]

print(f"train: {len(train_df):,} encounters of {len(np.unique(g_train)):,} patients | test: {len(test_df):,} encounters of {test_df[common.GROUP_COLUMN].nunique():,} patients")
print("patients in both sets:", len(set(g_train) & set(test_df[common.GROUP_COLUMN])))
display(pd.DataFrame({"train": y_train.value_counts(normalize=True).sort_index().values,
                      "test": y_test.value_counts(normalize=True).sort_index().values}, index=common.CLASS_LABELS).round(3))

## 5. Exploratory analysis (training patients only)

The questions: which inputs are related to an early readmission, is the relationship plausible, and what does the target look like across the inputs?

In [ ]:
feat_train = common.build_features(X_train)


def rate_table(values, y, min_n=200):
    """Share of each class among the encounters with each value of ``values`` (levels with fewer than min_n rows are dropped)."""
    frame = pd.DataFrame({"value": pd.Series(values).reset_index(drop=True), "y": pd.Series(y).reset_index(drop=True)})
    grouped = frame.groupby("value", observed=True)["y"]
    table = pd.DataFrame({
        "encounters": grouped.size(),
        "early <30": grouped.apply(lambda s: (s == 2).mean()),
        "late >30": grouped.apply(lambda s: (s == 1).mean()),
        "any readmission": grouped.apply(lambda s: (s != 0).mean()),
    })
    return table[table["encounters"] >= min_n]


def rate_plot(table, title, ax, order=None, rotate=0):
    table = table.loc[[o for o in order if o in table.index]] if order is not None else table   # levels below min_n were dropped
    positions = np.arange(len(table))
    ax.bar(positions - 0.2, table["early <30"], width=0.4, label="readmitted within 30 days (<30)", color="#c0392b")
    ax.bar(positions + 0.2, table["any readmission"], width=0.4, label="any readmission", color="#7f8c8d")
    ax.axhline((y_train == 2).mean(), color="#c0392b", linestyle="--", linewidth=1)
    ax.set_xticks(positions)
    ax.set_xticklabels([str(i) for i in table.index], rotation=rotate)
    ax.set_title(title)
    ax.set_ylabel("share of encounters")


fig, ax = plt.subplots(figsize=(6, 3.5))
share = y_train.value_counts(normalize=True).sort_index()
ax.bar(common.CLASS_LABELS, share.values, color=["#95a5a6", "#e67e22", "#c0392b"])
for i, value in enumerate(share.values):
    ax.text(i, value + 0.01, f"{value:.1%}", ha="center")
ax.set_title("Target distribution (training patients)")
ax.set_ylabel("share of encounters")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
table = rate_table(X_train["number_inpatient"].clip(upper=5), y_train)
table.index = [f"{int(i)}+" if i == 5 else str(int(i)) for i in table.index]
rate_plot(table, "Prior inpatient visits in the last year (5 = five or more)", axes[0])
axes[0].legend(fontsize=8)
table = rate_table(X_train["number_emergency"].clip(upper=4), y_train)
table.index = [f"{int(i)}+" if i == 4 else str(int(i)) for i in table.index]
rate_plot(table, "Prior emergency visits in the last year (4 = four or more)", axes[1])
plt.tight_layout()
plt.show()
display(rate_table(X_train["number_inpatient"].clip(upper=5), y_train).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
rate_plot(rate_table(X_train["age"], y_train, 100), "Age group", axes[0], order=[a for a in common.AGE_LABELS if a in set(X_train["age"])], rotate=45)
rate_plot(rate_table(X_train["time_in_hospital"], y_train, 100), "Length of stay (days)", axes[1])
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
table = rate_table(feat_train["diag_1_group"], y_train).sort_values("encounters", ascending=False)
rate_plot(table, "Primary diagnosis group", axes[0], rotate=45)
table = rate_table(feat_train["discharge_disposition"], y_train, 300).sort_values("encounters", ascending=False)
rate_plot(table, "Discharge disposition id (1 = home, 3 = skilled nursing, 6 = home health)", axes[1])
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
rate_plot(rate_table(feat_train["A1Cresult"], y_train), "HbA1c result ('None' = not measured)", axes[0], order=[v for v in common.A1C_VALUES if v in set(feat_train["A1Cresult"])])
rate_plot(rate_table(feat_train["insulin"], y_train), "Insulin", axes[1], order=[v for v in common.MED_STATUS if v in set(feat_train["insulin"])])
rate_plot(rate_table(feat_train["change"], y_train), "Diabetes medication changed?", axes[2])
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
table = rate_table(feat_train["medical_specialty"], y_train, 600).sort_values("encounters", ascending=False).head(10)
rate_plot(table, "Admitting specialty (10 most frequent)", axes[0], rotate=60)
table = rate_table(feat_train["payer_code"], y_train, 300).sort_values("encounters", ascending=False)
rate_plot(table, "Payer code", axes[1], rotate=45)
plt.tight_layout()
plt.show()

In [ ]:
numeric = feat_train[["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications", "number_diagnoses",
                      "n_prior_visits", "n_meds_active", "age_mid"]].copy()
numeric["class"] = train_df["readmitted"].values
display(numeric.groupby("class").mean().reindex(common.CLASS_LABELS).round(2).T)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, column in zip(axes, ["num_lab_procedures", "num_medications", "time_in_hospital"]):
    sns.boxplot(data=numeric, x="class", y=column, order=common.CLASS_LABELS, ax=ax, showfliers=False, palette=["#95a5a6", "#e67e22", "#c0392b"])
    ax.set_title(column)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = feat_train[common.NUMERIC_FEATURES].corr()
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f", annot_kws={"size": 7}, ax=ax, cbar=False)
ax.set_title("Correlation of the numeric features (training patients)")
plt.tight_layout()
plt.show()

In [ ]:
# Mutual information between every engineered feature and the target (a model-free screen, training patients only).
encoded = feat_train.copy()
for column in common.CATEGORICAL_FEATURES:
    encoded[column] = encoded[column].astype("category").cat.codes
encoded = encoded.fillna(encoded.median())
mi = mutual_info_classif(encoded.round().astype(int), y_train, discrete_features=True, random_state=RS)
mi = pd.Series(mi, index=encoded.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 6))
mi.head(15)[::-1].plot.barh(ax=ax, color="#2c3e50")
ax.set_title("Mutual information with the target (nats), top 15")
plt.tight_layout()
plt.show()

**EDA findings.**

* The first plot and the table below it show the strongest signal to be **past use of the hospital**: the share of early readmissions grows with the number of inpatient (and emergency) visits in the previous year.
  That is clinically sensible (frailty, unstable disease) and it is available at discharge.
* Age, length of stay, the primary diagnosis group, the discharge disposition and the admitting specialty show visible differences between levels in the plots above, but each effect is modest next to the history of hospital use.
* HbA1c measurement and drug changes show only small differences between the classes, in line with the small effect reported in the original study (Strack et al., 2014).
* The numeric inputs are only moderately correlated with each other (see the heatmap), so most of them carry some separate information.
* No single input separates the classes, so the achievable performance from these inputs is expected to be **moderate**. This is a hard, noisy problem, and the evaluation has to be honest about it.

## 6. Modelling approach

**Probabilities first.** All four models are trained on the natural class frequencies and output class probabilities. The model is selected by **cross-validated multi-class log-loss**, a proper scoring rule
that rewards honest, well-calibrated probabilities (which is what the web service returns). Threshold-free ROC-AUC and PR-AUC for the class `<30` are reported next to it.

**Decisions second.** With 11 % early readmissions, the arg-max of honest probabilities almost never says `<30`. The decision rule is therefore
`argmax(weight_c x probability_c)` with weight 1 for `NO` and two weights tuned on the **out-of-fold** predictions of the training patients to maximise the macro-F1 (section 7).
The weights move the operating point between recall and precision; they do not change the ranking quality of the model.

**Validation.** Patient-grouped, stratified 5-fold cross-validation on the training patients; the test patients are untouched until section 8.

In [ ]:
cv_folds = list(common.grouped_cv_splitter().split(np.zeros(len(y_train)), y_train, g_train))

# Reference: a predictor that always answers with the training class frequencies (no information at all).
reference = [common.probability_metrics(y_train.iloc[v], common.prior_probabilities(y_train.iloc[t], len(v))) for t, v in cv_folds]
reference_ll = np.mean([m["log_loss"] for m in reference])
print(f"Class-frequency reference: CV log-loss {reference_ll:.4f}, ROC-AUC 0.500 by construction")

# First model: logistic regression with default settings.
t0 = time.time()
cv_lr = common.cross_validate_model(common.build_pipeline("logistic_regression"), X_train, y_train, g_train)
s = cv_lr["summary"]
print(f"Logistic regression (default): CV log-loss {s['log_loss_mean']:.4f} +/- {s['log_loss_std']:.4f} | "
      f"AUC macro {s['roc_auc_macro_ovr_mean']:.3f} | AUC <30 {s['roc_auc_early_mean']:.3f} ({time.time() - t0:.0f}s)")

## 7. Hyper-parameter tuning

Tuning is the expensive part, so it runs on a **sample of whole patients** (about 30,000 encounters, 3 patient-grouped folds) and optimises the log-loss.
The starting values from `common.DEFAULT_PARAMS` are scored on the same folds; whichever is better is kept. Random search is used for the tree-based models, a small grid for the logistic regression.
The final comparison in section 7.2 uses all training patients and 5 folds.

In [ ]:
TUNE_ROWS = 3000 if FAST else 30000
tune_mask = common.sample_rows_by_group(g_train, TUNE_ROWS, seed=RS)
X_tune, y_tune, g_tune = X_train[tune_mask], y_train[tune_mask], g_train[tune_mask]
tune_cv = list(common.grouped_cv_splitter(n_splits=3).split(np.zeros(len(y_tune)), y_tune, g_tune))
print(f"tuning sample: {len(X_tune):,} encounters of {len(np.unique(g_tune)):,} patients")

defaults = {name: dict(params) for name, params in common.DEFAULT_PARAMS.items()}
SPACES = {
    "logistic_regression": ({"model__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]}, "grid", None),
    "decision_tree": ({"model__max_depth": [3, 4, 5, 6, 8, 10, 12], "model__min_samples_leaf": [20, 50, 100, 200, 400]}, "random", 10),
    "random_forest": ({"model__max_depth": [8, 10, 12, 14, 18], "model__min_samples_leaf": [10, 20, 30, 50, 100],
                       "model__max_features": ["sqrt", 0.15, 0.3], "model__max_samples": [0.3, 0.5, 0.8]}, "random", 6),
    "hist_gradient_boosting": ({"model__learning_rate": [0.03, 0.05, 0.08, 0.1], "model__max_iter": [100, 150, 200, 300],
                                "model__max_leaf_nodes": [7, 15, 31], "model__min_samples_leaf": [50, 100, 200, 400],
                                "model__l2_regularization": [0.0, 1.0, 10.0]}, "random", 6),
}
tuned_params, tuning_rows = {}, []


def tune(name):
    space, kind, n_iter = SPACES[name]
    pipeline = common.build_pipeline(name, defaults[name], n_jobs=N_JOBS)
    t0 = time.time()
    default_loss = -cross_val_score(pipeline, X_tune, y_tune, cv=tune_cv, scoring="neg_log_loss").mean()
    if kind == "grid":
        search = GridSearchCV(pipeline, space, scoring="neg_log_loss", cv=tune_cv, refit=False)
    else:
        search = RandomizedSearchCV(pipeline, space, n_iter=min(n_iter, 2) if FAST else n_iter, scoring="neg_log_loss",
                                    cv=tune_cv, random_state=RS, refit=False)
    search.fit(X_tune, y_tune)
    searched_loss = -search.best_score_
    if searched_loss < default_loss:
        tuned_params[name] = {**defaults[name], **common.plain_params(search.best_params_)}
    else:
        tuned_params[name] = defaults[name]
    tuning_rows.append({"model": name, "default log-loss": default_loss, "best searched log-loss": searched_loss,
                        "kept": "searched" if searched_loss < default_loss else "default",
                        "candidates": len(search.cv_results_["params"]), "seconds": time.time() - t0})
    print(f"{name}: default {default_loss:.4f} | best searched {searched_loss:.4f} -> {tuned_params[name]} ({time.time() - t0:.0f}s)")

In [ ]:
tune("logistic_regression")

In [ ]:
tune("decision_tree")

In [ ]:
tune("random_forest")

In [ ]:
tune("hist_gradient_boosting")

In [ ]:
display(pd.DataFrame(tuning_rows).set_index("model").round(4))

### 7.1 Full cross-validation of the four tuned models

All four models, tuned parameters, all training patients, 5 patient-grouped folds.

In [ ]:
cv_tuned, cv_summary = {}, {}
for name in common.MODEL_NAMES:
    t0 = time.time()
    cv_tuned[name] = common.cross_validate_model(common.build_pipeline(name, tuned_params[name], n_jobs=N_JOBS), X_train, y_train, g_train)
    cv_summary[name] = cv_tuned[name]["summary"]
    print(f"{name:<24s} done in {time.time() - t0:.0f}s")

comparison = pd.DataFrame(cv_summary).T[["log_loss_mean", "log_loss_std", "roc_auc_macro_ovr_mean", "roc_auc_early_mean", "pr_auc_early_mean"]]
comparison.loc["class-frequency reference"] = [reference_ll, np.std([m["log_loss"] for m in reference]), 0.5, 0.5, (y_train == 2).mean()]
comparison.columns = ["log-loss", "std", "ROC-AUC macro", "ROC-AUC <30", "PR-AUC <30"]
display(comparison.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
names = list(cv_summary)
axes[0].barh(names, [cv_summary[n]["log_loss_mean"] for n in names], xerr=[cv_summary[n]["log_loss_std"] for n in names], color="#2c3e50")
axes[0].axvline(reference_ll, color="#c0392b", linestyle="--", label="class-frequency reference")
axes[0].set_xlim(min(cv_summary[n]["log_loss_mean"] for n in names) - 0.02, reference_ll + 0.01)
axes[0].set_title("CV log-loss (lower is better)")
axes[0].legend(fontsize=8)
axes[1].barh(names, [cv_summary[n]["roc_auc_early_mean"] for n in names], xerr=[cv_summary[n]["roc_auc_early_std"] for n in names], color="#c0392b")
axes[1].axvline(0.5, color="black", linestyle="--")
axes[1].set_xlim(0.48, max(cv_summary[n]["roc_auc_early_mean"] for n in names) + 0.03)
axes[1].set_title("CV ROC-AUC for the early readmission <30")
plt.tight_layout()
plt.show()

### 7.2 Model selection and the decision weights

The model with the lowest cross-validated log-loss is selected (the test patients play no role). Its decision weights are tuned on its out-of-fold predictions.
For comparison, every model's own best macro-F1 on its out-of-fold predictions is shown too.

In [ ]:
selected = min(cv_summary, key=lambda n: cv_summary[n]["log_loss_mean"])
oof = {name: cv_tuned[name]["oof_proba"] for name in common.MODEL_NAMES}
weights_by_model, oof_f1 = {}, {}
for name in common.MODEL_NAMES:
    weights_by_model[name], oof_f1[name] = common.tune_decision_weights(y_train, oof[name])
print("Selected model (lowest CV log-loss):", selected)
display(pd.DataFrame({"decision weights (NO, >30, <30)": {n: weights_by_model[n] for n in weights_by_model},
                      "out-of-fold macro-F1": oof_f1}).round(3))

weights = weights_by_model[selected]
oof_metrics = common.decision_metrics(y_train, oof[selected], weights)
display(pd.DataFrame(oof_metrics["per_class"]).T.round(3))
print(f"Out-of-fold macro-F1 {oof_metrics['macro_f1']:.3f} | balanced accuracy {oof_metrics['balanced_accuracy']:.3f} | accuracy {oof_metrics['accuracy']:.3f}")

In [ ]:
# The trade-off behind the weight of the class <30: more weight finds more early readmissions but flags more patients wrongly.
rows = []
for w_early in common.WEIGHT_GRID_EARLY:
    m = common.decision_metrics(y_train, oof[selected], [1.0, weights[1], w_early])
    rows.append({"w": w_early, "recall": m["per_class"]["<30"]["recall"], "precision": m["per_class"]["<30"]["precision"], "macro_f1": m["macro_f1"],
                 "flagged": m["per_class"]["<30"]["predicted"] / len(y_train)})
trade = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(trade["w"], trade["recall"], label="recall of <30", color="#c0392b")
ax.plot(trade["w"], trade["precision"], label="precision of <30", color="#2980b9")
ax.plot(trade["w"], trade["macro_f1"], label="macro-F1 (all classes)", color="#27ae60")
ax.plot(trade["w"], trade["flagged"], label="share of encounters flagged", color="#7f8c8d", linestyle=":")
ax.axvline(weights[2], color="black", linestyle="--", linewidth=1, label=f"chosen weight {weights[2]:g}")
ax.set_xlabel("weight of the class <30"); ax.set_title(f"Operating point of {selected} (out-of-fold, training patients)")
ax.legend(fontsize=8)
plt.show()

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve, roc_curve

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for name in common.MODEL_NAMES:
    fpr, tpr, _ = roc_curve(y_train == 2, oof[name][:, 2])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC {cv_summary[name]['roc_auc_early_mean']:.3f})")
    precision, recall, _ = precision_recall_curve(y_train == 2, oof[name][:, 2])
    axes[1].plot(recall, precision, label=name)
axes[0].plot([0, 1], [0, 1], color="black", linestyle="--")
axes[0].set_title("ROC curve for <30 (out-of-fold)"); axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("true positive rate"); axes[0].legend(fontsize=8)
axes[1].axhline((y_train == 2).mean(), color="black", linestyle="--")
axes[1].set_title("Precision-recall curve for <30 (out-of-fold)"); axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
frac, mean_pred = calibration_curve(y_train == 2, oof[selected][:, 2], n_bins=10, strategy="quantile")
axes[2].plot(mean_pred, frac, marker="o", color="#c0392b", label=selected)
axes[2].plot([0, max(mean_pred.max(), frac.max())], [0, max(mean_pred.max(), frac.max())], color="black", linestyle="--", label="perfect calibration")
axes[2].set_title("Calibration of P(<30) (out-of-fold)"); axes[2].set_xlabel("mean predicted probability"); axes[2].set_ylabel("observed share"); axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 8. One-shot evaluation on the hold-out test patients

The selected model is fitted on the training patients with the tuned parameters and the decision weights from section 7.2, and is scored **once** on patients it has never seen.
The class-frequency reference is shown next to it.

In [ ]:
final_candidate = common.build_pipeline(selected, tuned_params[selected], n_jobs=N_JOBS).fit(X_train, y_train)
proba_test = common.predict_proba(final_candidate, X_test)
test_metrics = common.classification_metrics(y_test, proba_test, weights)
reference_test = common.classification_metrics(y_test, common.prior_probabilities(y_train, len(y_test)), [1.0, 1.0, 1.0])

summary = pd.DataFrame({selected: test_metrics, "class-frequency reference": reference_test}).T[
    ["log_loss", "roc_auc_macro_ovr", "roc_auc_early", "pr_auc_early", "roc_auc_any_readmission", "macro_f1", "balanced_accuracy", "accuracy"]]
display(summary.astype(float).round(3))
display(pd.DataFrame(test_metrics["per_class"]).T.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
cm = np.array(test_metrics["confusion_matrix"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=common.CLASS_LABELS, yticklabels=common.CLASS_LABELS, ax=axes[0], cbar=False)
axes[0].set_xlabel("predicted"); axes[0].set_ylabel("true"); axes[0].set_title("Confusion matrix (counts)")
sns.heatmap(cm / cm.sum(axis=1, keepdims=True), annot=True, fmt=".2f", cmap="Blues", xticklabels=common.CLASS_LABELS, yticklabels=common.CLASS_LABELS, ax=axes[1], cbar=False)
axes[1].set_xlabel("predicted"); axes[1].set_ylabel("true"); axes[1].set_title("Confusion matrix (row-normalised = recall)")
plt.tight_layout()
plt.show()

# How would a hospital use the ranking? Flag the highest-risk share of discharges and see how many early readmissions are caught.
order = np.argsort(-proba_test[:, 2])
early = (y_test.to_numpy() == 2)[order]
rows = []
for share in (0.05, 0.10, 0.20, 0.30, 0.50):
    k = int(round(share * len(order)))
    caught = early[:k].sum()
    rows.append({"flagged share": share, "flagged encounters": k, "early readmissions caught": int(caught),
                 "share of all early readmissions caught": caught / early.sum(), "precision": caught / k, "lift vs random": (caught / k) / early.mean()})
display(pd.DataFrame(rows).round(3))

## 9. Subgroup audit

`race` and `gender` are not inputs of the model, but a model can still behave differently across groups (through proxies such as payer, diagnosis or utilisation).
The table compares, for the test patients, the observed early-readmission rate with the mean predicted probability, the ranking quality (ROC-AUC for `<30`) and the share of early readmissions found by the decision rule.
Small groups are omitted because their estimates are too noisy.

In [ ]:
audit = test_df[["race", "gender", "age"]].copy()
audit["race"] = audit["race"].fillna("Not recorded")
audit["y"] = y_test.to_numpy()
audit["p_early"] = proba_test[:, 2]
audit["flag"] = common.decide(proba_test, weights) == 2


def audit_table(column, min_n=300 if not FAST else 50):
    rows = []
    for level, sub in audit.groupby(column):
        if len(sub) < min_n:
            continue
        positives = int((sub["y"] == 2).sum())
        rows.append({column: level, "encounters": len(sub), "observed <30 rate": positives / len(sub),
                     "mean predicted P(<30)": sub["p_early"].mean(), "ROC-AUC <30": common._auc(sub["y"] == 2, sub["p_early"]),
                     "recall <30": (sub["flag"] & (sub["y"] == 2)).sum() / max(1, positives), "share flagged": sub["flag"].mean()})
    return pd.DataFrame(rows).set_index(column).round(3)


for column in ("race", "gender", "age"):
    display(audit_table(column))

**Reading the audit.** Differences in the observed rate between groups reflect real differences in the data (and in who gets admitted); what matters for fairness is whether the model's *errors* differ.
If the mean predicted probability tracks the observed rate and the AUC and recall are similar across groups, the model is not systematically worse for one group. Gaps in `recall <30` or `share flagged`
would mean that a programme using the model would reach one group less often than another.
Even with a clean audit, the model was trained on 1999-2008 data from 130 US hospitals and must not be assumed to transfer to other populations or eras.

## 10. What drives the predictions, and how much a random split would flatter the model

### 10.1 Permutation importance on the test patients

Every raw input (or group of inputs) is shuffled across patients and the damage is measured: the increase of the log-loss and the drop of the ROC-AUC for `<30`.
The 23 diabetes drugs are shuffled together as one group. Larger means more important.

In [ ]:
n_imp = min(len(X_test), 1500 if FAST else 15000)
imp_idx = np.random.default_rng(RS).choice(len(X_test), n_imp, replace=False)
importance = common.grouped_permutation_importance(final_candidate, X_test.iloc[imp_idx].reset_index(drop=True),
                                                   y_test.iloc[imp_idx].reset_index(drop=True), n_repeats=2)
display(importance.round(4).head(12))
fig, ax = plt.subplots(figsize=(8, 6.5))
top = importance.head(15)[::-1]
ax.barh(top["variable"], top["log_loss_increase"], xerr=top["log_loss_increase_std"], color="#2c3e50")
ax.set_title(f"Permutation importance ({selected}, test patients)")
ax.set_xlabel("increase of the log-loss when the input is shuffled")
plt.tight_layout()
plt.show()

In [ ]:
# For interpretation: the largest coefficients of the (CV-tuned) logistic regression, in the language of "odds of an early readmission".
lr = common.build_pipeline("logistic_regression", tuned_params["logistic_regression"]).fit(X_train, y_train)
names = lr.named_steps["preprocess"].get_feature_names_out()
coef = pd.Series(lr.named_steps["model"].coef_[common.EARLY], index=names).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
pd.concat([coef.head(8), coef.tail(8)]).plot.barh(ax=ax, color=["#2980b9"] * 8 + ["#c0392b"] * 8)
ax.set_title("Logistic regression: largest coefficients for <30 versus NO (standardised numeric inputs)")
plt.tight_layout()
plt.show()

### 10.2 Patient leakage: what a random split would have reported

The same selected model is cross-validated on the tuning sample twice: with patient-grouped folds (what this project does) and with ordinary shuffled folds (which put a patient's other encounters into the training folds).

In [ ]:
t0 = time.time()
pipeline_leak = common.build_pipeline(selected, tuned_params[selected], n_jobs=N_JOBS)
grouped = common.cross_validate_model(pipeline_leak, X_tune, y_tune, g_tune, n_splits=5)["summary"]
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)    # StratifiedKFold ignores the groups: that is the point of this demonstration
    shuffled = common.cross_validate_model(pipeline_leak, X_tune, y_tune, g_tune,
                                           splitter=StratifiedKFold(5, shuffle=True, random_state=RS))["summary"]
leak = pd.DataFrame({"patient-grouped folds (used here)": grouped, "ordinary shuffled folds": shuffled}).T[
    ["log_loss_mean", "roc_auc_macro_ovr_mean", "roc_auc_early_mean", "pr_auc_early_mean"]]
leak.columns = ["log-loss", "ROC-AUC macro", "ROC-AUC <30", "PR-AUC <30"]
display(leak.round(4))
print(f"Difference grouped minus shuffled: log-loss {grouped['log_loss_mean'] - shuffled['log_loss_mean']:+.4f}, "
      f"ROC-AUC for <30 {grouped['roc_auc_early_mean'] - shuffled['roc_auc_early_mean']:+.4f} ({time.time() - t0:.0f}s). "
      "A positive log-loss difference (or a negative AUC difference) means that ordinary shuffled folds flatter the model.")

## 11. Write the tuned parameters for `train.py`

`train.py` reads `params.json`, repeats the cross-validation on all four models with exactly these parameters, selects the model, refits it on all patients and saves `model.bin`.

In [ ]:
meta = {
    "status": "tuned",
    "written_by": "notebook.ipynb",
    "tuning": f"3-fold patient-grouped search on about {len(X_tune):,} encounters, scoring = log-loss; starting values kept when the search did not beat them",
    "selected_in_notebook": selected,
    "decision_weights_in_notebook": weights,
    "cv_log_loss_in_notebook": {n: round(cv_summary[n]["log_loss_mean"], 4) for n in cv_summary},
    "versions": common.library_versions(),
}
common.save_params(PARAMS_PATH, tuned_params, meta)
print(f"Wrote {PARAMS_PATH.name}")
print(json.dumps({"selected": selected, "params": common.to_builtin(tuned_params[selected]), "decision_weights": weights}, indent=2))

## 12. Conclusions

* **The problem is hard, and the honest answer is a moderate one.** Read the AUC, precision-recall and lift numbers above for how well the best model ranks early readmissions from the information available at discharge; the permutation importance in section 10.1 shows which inputs it relies on.
* **The model is a triage tool, not a diagnosis.** The lift table in section 8 shows how many early readmissions are caught when only the highest-ranked share of discharges is flagged, which is how a transitional-care programme with limited capacity could use it.
  The decision weights are a business choice: raising the weight of `<30` finds more early readmissions at the price of more false alarms (section 7.2).
* **Validation matters.** Section 10.2 compares patient-aware folds with ordinary shuffled folds; every figure in this project comes from patient-aware validation.
* **Fairness and scope.** Race and gender are not inputs; the audit in section 9 checks that errors are not systematically different across groups. The data are from 130 US hospitals in 1999-2008 and cover only diabetes patients;
  the service is a decision-support demonstration and must not be used for clinical decisions.
* **Next steps:** more recent data, discharge notes and lab values rather than only counts, time-aware validation, calibration per hospital, and prospective evaluation before any use.